In [1]:
import os
import re
import contextlib
from unittest.mock import patch
import time

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from helpers import run_GP_and_Plot
import matlab.engine

torch.manual_seed(42)
np.random.seed(24601)

In [2]:
raw = pd.read_csv("VEGF_suppression_all.csv")

clean = raw.rename(columns=lambda x: re.sub(r'\s+', '_', x.lower().strip()))

for_bo = (
    clean[~clean['dataset'].str.startswith('Without_')]
    .pipe(lambda df: df[df['dose'] == 2.0])
    .query('k_d_value == 19000 and k_off_value == 0.864 and total_r2 <= 60e-4')
    .drop_duplicates(subset=['50%_retina'], keep='first')
    .assign(iteration=0)
)

for_bo.to_csv("dataset_for_bo.csv", index = False)
print("Saved dataset_for_bo.csv!")

Saved dataset_for_bo.csv!


In [3]:
eng = matlab.engine.start_matlab() # Initiating MATLAB connection

target_col = '50%_retina'
max_iterations = 20

In [4]:
start_time = time.time()

print(f"\nTarget: {target_col}")
print(f"\nRUNNING GP...")
print(f"\n{'=' * 60}")

for i in range(max_iterations):
    
    print(f"\nITERATION {i+1}")
    print(f"{'-' * 12}")
    
    df = pd.read_csv("dataset_for_bo.csv")
    
    with open(os.devnull, "w") as devnull:
        with contextlib.redirect_stdout(devnull): # to prevent function print statements
            with patch("matplotlib.pyplot.show"): # to prevent function plots
                
                next_point, ei_val, pred_mean, std, figs = run_GP_and_Plot(
                    df.rename(columns = {"total_r2": "Total Radius", 
                                        "fraction_pcl": "Fraction PCL"}),
                    feature_cols = ["Total Radius", "Fraction PCL"],
                    target_col = target_col,
                    bounds_dict = {
                        "Total Radius": (float(df["total_r2"].min()), 60e-4),
                        "Fraction PCL": (0.0, 1.0),
                    },
                    condn_col = "dose", condn_thresh = 2.0
                )
                
                plt.close('all')

    # Calculating inputs for the .m script
    r2 = next_point['Total Radius']
    frac_pcl = next_point['Fraction PCL']
    delr = r2 * frac_pcl
    r1 = r2 - delr
    thickness_scale = delr / ((12.7e-4)/2 - (10.2e-4)/2)
    radius_scale = r1 / (10.2e-4/2)

    # Inputs for the .m script
    eng.workspace['radius_scale'] = float(radius_scale)
    eng.workspace['thickness_scale'] = float(thickness_scale)
    eng.workspace['dose_in'] = 2.0
    
    eng.workspace['csv_name'] = 'dataset_for_bo.csv'

    # Running the .m script to add a new row to the dataset
    eng.eval("run('ranibizumab_dds_one_case_for_bo.m')", nargout = 0)

    # Printing observations for current iteration
    df = pd.read_csv("dataset_for_bo.csv")
    bo_df = df[df["iteration"] != 0]
    current_iter_val = bo_df[bo_df["iteration"] == bo_df["iteration"].max()][target_col].values[0]
    best_observed = bo_df[target_col].max()
    
    print(f"GP predicted: {pred_mean:.3f}")
    print(f"Current iteration: {current_iter_val:.3f}")
    print(f"Best observed so far: {best_observed:.3f}")
    
    # Convergence check: GP predicted < best observed
    if pred_mean < best_observed:
        print(f"\n{'=' * 60}")
        print(f"\nStopping!\nGP predicted {pred_mean:.3f} < best observed {best_observed:.3f}")
        break
    
    # Convergence check: target value at 1dp appears 3+ times
    bo_vals = bo_df[target_col].round(1)
    if bo_vals.value_counts().max() >= 3:
        print(f"\n{'=' * 50}")
        print(f"\nStopping!\n{target_col} repeated 3+ times!")
        break

print(f"\nBest observed {target_col}: {best_observed:.3f}")
print(f"\n{'=' * 50}")
print(f"\nLOOP FINISHED!!")

eng.quit() # Closing MATLAB connection

elapsed = time.time() - start_time
print(f"\n\nTime taken: {elapsed/60:.1f} minutes")


Target: 50%_retina

RUNNING GP...


ITERATION 1
------------
GP predicted: 437.953
Current iteration: 424.217
Best observed so far: 424.217

ITERATION 2
------------
GP predicted: 431.942
Current iteration: 424.417
Best observed so far: 424.417

ITERATION 3
------------
GP predicted: 429.776
Current iteration: 424.617
Best observed so far: 424.617

ITERATION 4
------------
GP predicted: 428.652
Current iteration: 424.617
Best observed so far: 424.617

ITERATION 5
------------
GP predicted: 427.931
Current iteration: 424.817
Best observed so far: 424.817

ITERATION 6
------------
GP predicted: 427.432
Current iteration: 424.717
Best observed so far: 424.817

ITERATION 7
------------
GP predicted: 425.526
Current iteration: 423.917
Best observed so far: 424.817

ITERATION 8
------------
GP predicted: 426.566
Current iteration: 424.717
Best observed so far: 424.817

ITERATION 9
------------
GP predicted: 426.380
Current iteration: 424.917
Best observed so far: 424.917

ITERATION 10
-----